# Control Evaluator

> Fill in a module description here

In [ ]:
#| default_exp evaluators.control

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from collections import defaultdict
import torch
import torch.nn as nn
import torch.nn.functional as F

from einops import rearrange
import hydra
import wandb
from fastcore.utils import patch
from c3jepa_wm.utils import channel

In [ ]:
#| export
class MultiAgentGoalEvaluator:
    """
    Dataset-driven evaluation of the JEPA planner for a 2-agent communicative setting.

    For each episode and each agent:
      - pick a start step `t0` (e.g. random or fixed)
      - the agent's goal is its own observation at `t0 + goal_offset`
      - the *other* agent encodes its own current observation via the VQ-VAE
        into `msg_indices` and "sends" it (used as conditioning for this agent's rollout)
      - run the planner to get a sequence of actions
      - compare against ground-truth actions / measure goal-reaching error

    Args:
        model: JEPA model (has .encoder, .predictor, .action_encoder, .get_cost, .rollout)
        vqvae: sender-side VQ-VAE module; vqvae.encode(pixels) -> (B, 1, 49) token indices
        planner_cls: planner class (e.g. JEPAGoalPlanner)
        planner_kwargs: kwargs to construct one planner per agent
        history_size: number of past steps fed as context (matches JEPA.rollout's history_size)
        goal_offset: number of steps ahead the goal is set
        agents: list of agent keys, must have length 2
        device: torch device
    """

    def __init__(
        self,
        data_module,
        model,
        planner_cfg,
        slurm_jobid= None,
        history_size=3,
        goal_offset=10,
        agents=("agent_0", "agent_1"),
        device="cpu",
        SNR = 10.0,  # example SNR for power/schedule extraction; in practice, this could be tuned or provided as part of the episode
        p_max = 10,  # max power level for scheduling decisions
        noise_power=1e-3,  # noise power for SNR calculations
        agent_env_idx=None,  # optional mapping from agent keys to environment indices
    ):
        assert len(agents) == 2, "This evaluator only supports exactly 2 agents."
        self.data_module = data_module
        self.data_module.setup()

        self.model = model['jepa'].to(device).eval()
        self.vqvae = model['vqvae'].to(device).eval()
        self.agents = list(agents)
        self.agent_env_idx = agent_env_idx or {a: i for i, a in enumerate(self.agents)}
        self.history_size = history_size
        self.goal_offset = goal_offset
        self.device = device
        self.SNR = SNR
        self.p_max = p_max
        self.noise_power = noise_power

        # one planner per agent (mirrors the per-agent action_dim/horizon if they differ)
        self.planners = {
            # agent: planner_cls(model=self.model, device=device, **planner_kwargs)
            agent: hydra.utils.instantiate(planner_cfg, model= self.model, device=device)
            for agent in self.agents
        }

    @torch.no_grad()
    def _encode_message(self, partner_pixels_vqvae_t0, csi_t0, schedule=None, power=None, no_comm=False):
        """
        partner_pixels_vqvae_t0: (B, C, H, W) -- partner's obs at t0, VQ-VAE-transform space
        csi_t0: (1,) complex -- channel state at t0 for this sender->receiver link
        schedule, power: scalars or (1,) tensors, or None to skip channel entirely
        """
        partner_pixels_vqvae_t0 = partner_pixels_vqvae_t0.to(self.device)
        indices = self.vqvae.get_message_indices(partner_pixels_vqvae_t0)  # (B, 1, H, W)
        indices = rearrange(indices, "B H W -> B (H W)").long() # (B, 49)

        if schedule is not None and power is not None:
            n = 1  # single neighbor (the other agent)
            # print(csi_t0.shape, schedule.shape, power.shape)
            csi = csi_t0.to(self.device) #csi_t0.reshape(1, n, 1).to(self.device)              # (B*T=1, n, 1) complex
            schedule = torch.as_tensor(schedule, device=self.device)#.reshape(1, n, 1)
            power = torch.as_tensor(power, device=self.device)#.reshape(1, n, 1)

            indices = channel(
                schedule=schedule,
                power=power,
                msg_indices=indices,       # (B*T, 49) -> here B*T = 1
                csi=csi,
                device=self.device,
                snr_db=self.SNR,
                no_comm=no_comm,
            )  # returns (B*T, 49) = (1, 49)

        return indices.unsqueeze(1)  # (1, 1, 49) -- matches msg_indices shape expected downstream
        
    def _extract_power_and_schedule(self, csi):
        snr_linear = 10 ** (self.SNR / 10.0)
        optimal_power = snr_linear * self.noise_power / (torch.abs(csi) ** 2 + 1e-8)
        schedule = (optimal_power <= self.p_max).float()
        return optimal_power, schedule
        
    # def _build_agent_info(self, episode, agent, partner, t0):
    #     import ipdb; ipdb.set_trace()
    #     H = self.history_size
    #     pixels = episode[agent]["pixels"][:, t0 - H + 1 : t0 + 1]
    #     actions = episode[agent]["action"][:, t0 - H + 1 : t0 + 1]
    #     goal_pixels = episode[agent]["pixels"][:, t0 + self.goal_offset]
    #     print(f"Agent {agent} at t0={t0}: pixels shape {pixels.shape}, actions shape {actions.shape}, goal_pixels shape {goal_pixels.shape}")

    #     partner_obs_vqvae_t0 = episode[partner]["pov_seq_vqvae"][:, t0]#.unsqueeze(0)
    #     csi_t0 = episode[partner]["csi"][:, t0]#.unsqueeze(0)  # (1,) complex -- confirm this is the right link's CSI
    #     csi_t0 = csi_t0.to(self.device)
    #     print("Getting the power level and schedule for the channel...")
    #     power_level, schedule = self._extract_power_and_schedule(csi_t0)
    #     print("Encoding message from partner's observation...")
    #     msg_indices = self._encode_message(
    #         partner_obs_vqvae_t0, csi_t0, schedule= schedule, power= power_level
    #     )

    #     info = {
    #         "pixels": pixels.to(self.device),
    #         "action": actions.to(self.device),
    #         "goal": goal_pixels.to(self.device),
    #         "msg_indices": msg_indices.to(self.device),
    #     }
    #     print(f"Built info dict for agent {agent}:")
    #     for k, v in info.items():
    #         print(f"  {k}: shape {v.shape}, dtype {v.dtype}")
    #     return info



    def _build_agent_info(self, episode, agent, partner, t0, goal_override=None):
        # import ipdb; ipdb.set_trace()
        H = self.history_size
        pixels = episode[agent]["pixels"][:, t0 - H + 1 : t0 + 1]
        actions = episode[agent]["action"][:, t0 - H + 1 : t0 + 1]
        goal_pixels = goal_override if goal_override is not None else episode[agent]["pixels"][:, t0 + self.goal_offset]

        partner_obs_vqvae_t0 = episode[partner]["pov_seq_vqvae"][:, t0]
        csi_t0 = episode[partner]["csi"][:, t0].to(self.device)
        power_level, schedule = self._extract_power_and_schedule(csi_t0)
        msg_indices = self._encode_message(partner_obs_vqvae_t0, csi_t0, schedule=schedule, power=power_level)

        info = {
            "pixels": pixels.to(self.device),
            "action": actions.to(self.device),
            "goal": goal_pixels.to(self.device),
            "msg_indices": msg_indices.to(self.device),
        }
        return info


    @torch.no_grad()
    def evaluate_episode(self, episode, t0=None):
        """
        episode: dict keyed by agent -> {"pixels": (B,T,C,H,W), "action": (B,T,action_dim)}
        t0: start step; if None, picked so that t0 - H + 1 >= 0 and t0 + goal_offset < T
        Returns per-agent dict with planned actions, ground-truth actions, and goal error.
        """
        H = self.history_size
        T = episode[self.agents[0]]["pixels"].size(1)
        lo = H - 1
        hi = T - self.goal_offset - 1
        assert hi >= lo, "Episode too short for given history_size/goal_offset."
        if t0 is None:
            t0 = torch.randint(lo, hi + 1, (1,)).item()

        results = {}
        for agent in self.agents:
            partner = [a for a in self.agents if a != agent][0]
            info = self._build_agent_info(episode, agent, partner, t0)
            print(f"Evaluating agent {agent} at t0={t0} with info keys: {list(info.keys())}")
            first_action, plan = self.planners[agent].plan(info)
            print(f"Agent {agent} planned actions shape: {plan.shape}, first action shape: {first_action.shape}")

            gt_future_actions = episode[agent]["action"][:, t0 + 1 : t0 + 1 + plan.size(1)]
            goal_err = self.planners[agent].eval_plan(info, self.device, plan) #self._goal_error(episode, agent, t0, plan)
            
            results[agent] = {
                "t0": t0,
                "planned_actions": plan.squeeze(0).cpu(),
                "first_action": first_action.cpu(),
                "gt_actions": gt_future_actions.cpu(),
                "goal_error": goal_err,
            }
        return results

    @torch.no_grad()
    def evaluate_dataset(self, num_episodes=None):
        """
        dataset: iterable of episodes (each a dict as in evaluate_episode)
        Returns a list of per-episode results plus aggregate stats.
        """
        dataset = self.data_module.val_dataloader()
        all_results = []
        # import ipdb; ipdb.set_trace()
        for i, episode in enumerate(dataset):
            if num_episodes is not None and i >= num_episodes:
                break
            all_results.append(self.evaluate_episode(episode))

            # if i > 1: # TODO: remove this
            #     break  # for debugging, limit to a few episodes

        agg = {agent: {"goal_error": []} for agent in self.agents}
        for ep_res in all_results:
            for agent in self.agents:
                agg[agent]["goal_error"].append(ep_res[agent]["goal_error"])

        summary = {}
        for agent in self.agents:
            # Stack the list of [2D tensors] into a single 2D tensor [N, 2]
            stacked_errors = torch.stack(agg[agent]["goal_error"])
            
            # Option A: If you want a single scalar distance error per agent (L2 Norm)
            # (Highly recommended for trajectory/goal error evaluation)
            l2_errors = torch.norm(stacked_errors, dim=-1) # Shape: [N]
            
            summary[agent] = {
                "mean_goal_error": float(l2_errors.mean()),
                "std_goal_error": float(l2_errors.std()),
                "n": len(agg[agent]["goal_error"]),
            }

        # Flatten the dictionary for WandB tracking
        wandb_log_dict = {
            f"eval/{agent}/{metric}": value
            for agent, metrics in summary.items()
            for metric, value in metrics.items()
        }

        # Log the flat dictionary
        # If running this at the end of an epoch, pass the current epoch/step if you have it
        wandb.log(wandb_log_dict)

        return {"per_episode": all_results, "summary": summary}
    

In [ ]:
#| export
def set_env_state(env, goal_pos, agent_positions, agent_directions, agent_env_indices):
    """
    Directly overwrites env state to match a recorded episode's configuration
    at some timestep, bypassing place_agent's random placement.

    agent_positions: dict[env_idx -> (x, y)]
    agent_directions: dict[env_idx -> int]
    """
    unwrapped = env.unwrapped
    goal_pos = tuple(int(v) for v in goal_pos)

    # Move the goal object in the grid to match the recorded position
    # (goal placement is stochastic per-episode, so a fresh reset(seed=0)
    # will NOT put it here on its own).
    old_goal_pos = tuple(unwrapped.goal_pos)
    if old_goal_pos != goal_pos:
        goal_obj = unwrapped.grid.get(*old_goal_pos)
        unwrapped.grid.set(*old_goal_pos, None)
        unwrapped.grid.set(*goal_pos, goal_obj)
    unwrapped.goal_pos = goal_pos

    # Same direct-assignment pattern as place_agent, just skipping the
    # random pos/dir draw since we already know the target values.
    for idx in agent_env_indices:
        agent = unwrapped.agents[idx]
        agent.state.pos = tuple(int(v) for v in agent_positions[idx])
        agent.state.dir = int(agent_directions[idx])
        

In [ ]:
#| export
@patch
@torch.no_grad()
def evaluate_episode_fixed_t0(self: MultiAgentGoalEvaluator, episode, env, t0, max_steps=150):
    """
    Evaluate one episode from a single fixed t0, using receding-horizon MPC:
    at every real environment step, run a fresh CEM plan (its own internal
    opt_steps x horizon optimization) and execute only the first action.
    Repeats for up to max_steps real steps (or until goal reached).

    Messaging: partner message is used only for the very first planning call
    at t0; dropped for all subsequent replanning calls.
    """
    H = self.history_size

    assert t0 >= H - 1, f"t0={t0} too early for history_size={H}"

    # --- fixed goal embeddings ---
    goal_emb = {}
    # import ipdb; ipdb.set_trace()
    for ai, agent in enumerate(self.agents):
        g = episode["goal_obs"][ai].unsqueeze(0).unsqueeze(0).to(self.device)
        enc = self.model.encode({"pixels": g})
        goal_emb[agent] = enc["emb"][:, 0]

    # --- set env to recorded state at t0 ---
    goal_pos = episode["goal_pos"].tolist()#episode["goal_pos"].tolist()
    agent_positions_t0 = {self.agent_env_idx[a]: episode[a]["pos"][0, t0].tolist() for a in self.agents}
    agent_directions_t0 = {self.agent_env_idx[a]: int(episode[a]["dir"][0, t0].item()) for a in self.agents}
    env.reset(seed=0)
    set_env_state(env, goal_pos, agent_positions_t0, agent_directions_t0, list(self.agent_env_idx.values()))

    # --- initial per-agent info (includes message, used once) ---
    info = {}
    for agent in self.agents:
        partner = [a for a in self.agents if a != agent][0]
        ai = self.agents.index(agent)
        goal_frame = episode["goal_obs"][ai] if episode["goal_obs"].dim() == 5 else episode["goal_obs"][ai:ai+1]
        info[agent] = self._build_agent_info(episode, agent, partner, t0, goal_override=goal_frame)

    results = {agent: {"step": [], "embedding_dist_to_goal": [], "reached": []} for agent in self.agents}
    active = {agent: True for agent in self.agents}  # stop updating an agent once it's reached goal

    for real_step in range(max_steps):
        step_actions = {}
        for agent in self.agents:
            if not active[agent]:
                step_actions[self.agent_env_idx[agent]] = 3  # no-op / stay, once done -- confirm action id with your env
                continue
            _, plan = self.planners[agent].plan(info[agent])  # full CEM: opt_steps iters over planner.horizon lookahead
            step_actions[self.agent_env_idx[agent]] = int(plan[0, 0].item())  # execute ONLY first action (receding horizon)

        obs = env.step(step_actions)[0]

        for agent in self.agents:
            if not active[agent]:
                continue
            idx = self.agent_env_idx[agent]
            pov = obs[idx]["pov"]
            frame = self.data_module.val_receiver_transform(pov).unsqueeze(0).unsqueeze(0).to(self.device)
            enc = self.model.encode({"pixels": frame})
            cur_emb = enc["emb"][:, 0]

            dist = F.mse_loss(cur_emb, goal_emb[agent], reduction="none").sum(-1)
            reached = tuple(env.unwrapped.agents[idx].state.pos) == tuple(goal_pos)

            results[agent]["step"].append(real_step)
            results[agent]["embedding_dist_to_goal"].append(dist.item())
            results[agent]["reached"].append(reached)

            new_action = torch.tensor([[step_actions[idx]]], device=self.device)
            info[agent] = {
                **info[agent],
                "pixels": torch.cat([info[agent]["pixels"][:, 1:], frame], dim=1),
                "action": torch.cat([info[agent]["action"][:, 1:], new_action], dim=1),
            }
            info[agent].pop("msg_indices", None)

            if reached:
                active[agent] = False

        if not any(active.values()):
            break

    return results

In [ ]:
#| export
@patch
@torch.no_grad()
def evaluate_dataset_fixed_t0(self: MultiAgentGoalEvaluator, env,
                               num_episodes=None, max_steps=150, t0=None):
    """
    Runs evaluate_episode_fixed_t0 (receding-horizon MPC, real-env execution)
    over multiple episodes from the val dataset, aggregating embedding
    distance-to-goal and success by real step index across episodes.

    t0: if None, defaults to history_size - 1 per episode (start MPC as soon
        as enough history exists). If an int, used as a fixed absolute t0
        for every episode (episodes too short for it are skipped).

    NOTE: assumes the dataloader yields batch_size=1 (single episode per
    batch), since only one episode's state can be loaded into `env` at a time.
    """
    
    dataset = self.data_module.val_dataloader()
    assert dataset.batch_size == 1, "evaluate_dataset_fixed_t0 requires batch_size=1"

    per_agent_step_dist = {agent: defaultdict(list) for agent in self.agents}
    per_agent_step_success = {agent: defaultdict(list) for agent in self.agents}

    n_evaluated = 0
    for i, batch in enumerate(dataset):
        if num_episodes is not None and n_evaluated >= num_episodes:
            break

        # strip the batch dim (B=1) to get a plain per-episode dict
        episode = {
            "goal_pos": batch["goal_pos"][0],
            "goal_obs": batch["goal_obs"][0],
        }
        for agent in self.agents:
            episode[agent] = batch[agent] #{k: v[0] for k, v in batch[agent].items()}

        length = batch["length"][0].item()
        this_t0 = (self.history_size - 1) if t0 is None else t0
        if this_t0 >= length:
            print(f"Skipping episode {i}: length={length} too short for t0={this_t0}")
            continue

        print(f"Evaluating episode {i} (fixed t0={this_t0}, max_steps={max_steps})...")
        ep_res = self.evaluate_episode_fixed_t0(
            episode, env=env, t0=this_t0, max_steps=max_steps
        )
        n_evaluated += 1

        for agent in self.agents:
            for step, dist, reached in zip(
                ep_res[agent]["step"], ep_res[agent]["embedding_dist_to_goal"], ep_res[agent]["reached"]
            ):
                per_agent_step_dist[agent][step].append(dist)
                per_agent_step_success[agent][step].append(float(reached))

    curves = {}
    for agent in self.agents:
        steps = sorted(per_agent_step_dist[agent].keys())
        curves[agent] = {
            "step": steps,
            "mean_embedding_dist": [float(torch.tensor(per_agent_step_dist[agent][s]).mean()) for s in steps],
            "std_embedding_dist": [float(torch.tensor(per_agent_step_dist[agent][s]).std()) for s in steps],
            # fraction of (still-running) episodes that have reached goal by this step
            "success_rate": [float(torch.tensor(per_agent_step_success[agent][s]).mean()) for s in steps],
        }

    for agent in self.agents:
        c = curves[agent]
        wandb.log({
            f"eval/{agent}/embedding_dist_vs_realstep": wandb.plot.line(
                wandb.Table(data=list(zip(c["step"], c["mean_embedding_dist"])), columns=["step", "mean_dist"]),
                x="step", y="mean_dist", title=f"{agent} embedding distance to goal vs real step (fixed t0)",
            ),
            f"eval/{agent}/success_rate_vs_realstep": wandb.plot.line(
                wandb.Table(data=list(zip(c["step"], c["success_rate"])), columns=["step", "success_rate"]),
                x="step", y="success_rate", title=f"{agent} cumulative success rate vs real step (fixed t0)",
            ),
        })

    return curves

In [ ]:
# #| export
# @patch
# @torch.no_grad()
# def evaluate_episode_over_time(self: MultiAgentGoalEvaluator, episode, t0_values=None, t0_stride=5):
#     """
#     Like evaluate_episode, but runs the planner at multiple t0 values across
#     the episode instead of a single random one, so you can track how goal
#     error evolves over the course of an episode.

#     t0_values: explicit list of t0s to evaluate; if None, generated via t0_stride
#                over the valid range [H-1, T-goal_offset-1].
#     Returns: {agent: {"t0": [...], "goal_error": [...]}}
#     """
#     H = self.history_size
#     T = episode[self.agents[0]]["pixels"].size(1)
#     lo = H - 1
#     hi = T - self.goal_offset - 1
#     assert hi >= lo, "Episode too short for given history_size/goal_offset."

#     if t0_values is None:
#         t0_values = list(range(lo, hi + 1, t0_stride))

#     results = {agent: {"t0": [], "goal_error": []} for agent in self.agents}
#     for t0 in t0_values:
#         for agent in self.agents:
#             partner = [a for a in self.agents if a != agent][0]
#             info = self._build_agent_info(episode, agent, partner, t0)
#             print("Planning for agent {} at t0={}".format(agent, t0))
#             first_action, plan = self.planners[agent].plan(info)
#             print("planned actions for agent {} at t0={}".format(agent, t0))
#             goal_err = self.planners[agent].eval_plan(info, self.device, plan)  # (B,) or (B, D)
#             print("computing the goal ERR...")

#             results[agent]["t0"].append(t0)
#             print("Appending goal error for agent {} at t0={}".format(agent, t0))
#             results[agent]["goal_error"].append(goal_err.cpu())
#             print("finished evaluating agent {} at t0={}".format(agent, t0))

#     return results

In [ ]:
#| export
@patch
@torch.no_grad()
def evaluate_episode_over_time(self: MultiAgentGoalEvaluator, episode, t0_values=None, t0_stride=5):
    H = self.history_size
    T = episode[self.agents[0]]["pixels"].size(1)
    lo = H - 1
    hi = T - self.goal_offset - 1
    assert hi >= lo, "Episode too short for given history_size/goal_offset."

    if t0_values is None:
        t0_values = list(range(lo, hi + 1, t0_stride))

    results = {agent: {"t0": [], "goal_error": [], "gt_success_by_target": []} for agent in self.agents}
    for t0 in t0_values:
        for agent in self.agents:
            partner = [a for a in self.agents if a != agent][0]
            info = self._build_agent_info(episode, agent, partner, t0)
            first_action, plan = self.planners[agent].plan(info)
            goal_err = self.planners[agent].eval_plan(info, self.device, plan)

            # Ground-truth statistic: had the recorded episode already reached its
            # success condition by the target timestep (t0 + goal_offset)?
            # NOTE: this reflects the *demonstration trajectory*, not the planner's
            # own chosen actions -- it's a difficulty/context label, not a
            # planner-success metric.
            success_at = episode[agent]["success_at"]  # (B,)
            target_t = t0 + self.goal_offset
            gt_success = (success_at <= target_t)      # (B,) bool

            results[agent]["t0"].append(t0)
            results[agent]["goal_error"].append(goal_err.cpu())
            results[agent]["gt_success_by_target"].append(gt_success.cpu())

    return results

In [ ]:
#| export
@patch
@torch.no_grad()
def evaluate_dataset_over_time(self: MultiAgentGoalEvaluator, num_episodes=None, t0_stride=5):
    dataset = self.data_module.val_dataloader()

    # separate error accumulators for "already succeeded in demo by target time"
    # vs "not yet succeeded" -- purely a data-driven stratification
    per_agent_t0_errors = {agent: defaultdict(list) for agent in self.agents}
    per_agent_t0_errors_success = {agent: defaultdict(list) for agent in self.agents}
    per_agent_t0_errors_pending = {agent: defaultdict(list) for agent in self.agents}
    per_agent_t0_success_rate = {agent: defaultdict(list) for agent in self.agents}  # fraction succeeded, per t0

    for i, episode in enumerate(dataset):
        if num_episodes is not None and i >= num_episodes:
            break
        print(f"Evaluating episode {i} over time...")
        ep_res = self.evaluate_episode_over_time(episode, t0_stride=t0_stride)

        for agent in self.agents:
            for t0, err, gt_success in zip(
                ep_res[agent]["t0"], ep_res[agent]["goal_error"], ep_res[agent]["gt_success_by_target"]
            ):
                # err, gt_success are both (B,) here (or (B, D) for err) -- reduce per-episode-batch to scalars
                err_per_sample = torch.norm(err, dim=-1) if err.dim() > 1 else err  # (B,)

                per_agent_t0_errors[agent][t0].append(err_per_sample.mean().item())
                per_agent_t0_success_rate[agent][t0].append(gt_success.float().mean().item())

                succ_mask = gt_success
                if succ_mask.any():
                    per_agent_t0_errors_success[agent][t0].append(err_per_sample[succ_mask].mean().item())
                if (~succ_mask).any():
                    per_agent_t0_errors_pending[agent][t0].append(err_per_sample[~succ_mask].mean().item())

    curves = {}
    for agent in self.agents:
        t0s = sorted(per_agent_t0_errors[agent].keys())
        curves[agent] = {
            "t0": t0s,
            "mean": [float(torch.tensor(per_agent_t0_errors[agent][t0]).mean()) for t0 in t0s],
            "std": [float(torch.tensor(per_agent_t0_errors[agent][t0]).std()) for t0 in t0s],
            "gt_success_rate": [float(torch.tensor(per_agent_t0_success_rate[agent][t0]).mean()) for t0 in t0s],
            "mean_when_gt_success": [
                float(torch.tensor(v).mean()) if (v := per_agent_t0_errors_success[agent].get(t0, [])) else float("nan")
                for t0 in t0s
            ],
            "mean_when_gt_pending": [
                float(torch.tensor(v).mean()) if (v := per_agent_t0_errors_pending[agent].get(t0, [])) else float("nan")
                for t0 in t0s
            ],
        }

    for agent in self.agents:
        c = curves[agent]
        wandb.log({
            f"eval/{agent}/goal_error_over_time": wandb.plot.line(
                wandb.Table(data=list(zip(c["t0"], c["mean"])), columns=["t0", "mean_goal_error"]),
                x="t0", y="mean_goal_error", title=f"{agent} goal error vs t0",
            ),
            f"eval/{agent}/gt_success_rate_over_time": wandb.plot.line(
                wandb.Table(data=list(zip(c["t0"], c["gt_success_rate"])), columns=["t0", "gt_success_rate"]),
                x="t0", y="gt_success_rate", title=f"{agent} ground-truth demo success rate by target time",
            ),
        })

    return curves

In [ ]:
# #| export
# @patch
# @torch.no_grad()
# def evaluate_dataset_over_time(self: MultiAgentGoalEvaluator, num_episodes=None, t0_stride=5):
#     dataset = self.data_module.val_dataloader()

#     # collect per-agent: {t0: [errors across episodes]}
#     per_agent_t0_errors = {agent: defaultdict(list) for agent in self.agents}

#     for i, episode in enumerate(dataset):
#         if num_episodes is not None and i >= num_episodes:
#             break
#         print(f"Evaluating episode {i} over time...")
#         ep_res = self.evaluate_episode_over_time(episode, t0_stride=t0_stride)
#         print(f"Episode {i} evaluation complete. Aggregating results...")
#         # import ipdb; ipdb.set_trace()
#         for agent in self.agents:
#             for t0, err in zip(ep_res[agent]["t0"], ep_res[agent]["goal_error"]):
#                 # err may be (B,) or (B, D) -- reduce to scalar per episode/t0 first
#                 err_scalar = torch.norm(err, dim=-1).mean().item() if err.dim() > 1 else err.mean().item()
#                 per_agent_t0_errors[agent][t0].append(err_scalar)

#     # build mean/std curves per agent
#     curves = {}
#     for agent in self.agents:
#         t0s = sorted(per_agent_t0_errors[agent].keys())
#         means = [float(torch.tensor(per_agent_t0_errors[agent][t0]).mean()) for t0 in t0s]
#         stds = [float(torch.tensor(per_agent_t0_errors[agent][t0]).std()) for t0 in t0s]
#         curves[agent] = {"t0": t0s, "mean": means, "std": stds}

#     # log as a wandb line plot -- one line per agent
#     for agent in self.agents:
#         table = wandb.Table(
#             data=list(zip(curves[agent]["t0"], curves[agent]["mean"])),
#             columns=["t0", "mean_goal_error"],
#         )
#         wandb.log({
#             f"eval/{agent}/goal_error_over_time": wandb.plot.line(
#                 table, x="t0", y="mean_goal_error",
#                 title=f"{agent} goal error vs t0",
#             )
#         })

#     return curves

### Goal-reaching inside the simulator

In [ ]:
#| export
def get_all_agent_plans(evaluator, episode, t0):
    """
    Run each agent's own CEM planner independently (mirrors evaluate_episode's
    per-agent loop), and return their plans keyed by agent.

    Returns:
        plans: dict[agent -> (B, horizon) long tensor of planned action indices]
        infos: dict[agent -> info_dict used to build that plan] (kept around in
               case you need it for eval_plan / goal_error too)
    """
    plans = {}
    infos = {}
    for agent in evaluator.agents:
        partner = [a for a in evaluator.agents if a != agent][0]
        info = evaluator._build_agent_info(episode, agent, partner, t0)
        first_action, plan = evaluator.planners[agent].plan(info)  # plan: (B, horizon)
        plans[agent] = plan
        infos[agent] = info
    return plans, infos

In [ ]:
#| export
def build_joint_step_actions(plans, agent_order, batch_idx=0):
    """
    plans: dict[agent -> (B, horizon) long tensor], one entry per agent
    agent_order: list of agent keys matching env's agent indexing, e.g. [0, 1]
                 (must match the order env.unwrapped.agents expects, NOT the
                 string keys like "agent_0" used elsewhere in this codebase)
    batch_idx: which batch element to extract (planning is usually done with
               B=1 per episode in this evaluator, so default 0)

    Returns:
        list of length `horizon`, each element a dict {agent_env_idx: action_int}
        ready to pass to env.step(...)
    """
    horizon = next(iter(plans.values())).size(1)
    step_actions = []
    for t in range(horizon):
        step_dict = {}
        for agent_key, env_idx in zip(plans.keys(), agent_order):
            step_dict[env_idx] = int(plans[agent_key][batch_idx, t].item())
        step_actions.append(step_dict)
    return step_actions

In [ ]:
#| export
def check_agent_reached_goal(env, agent_idx, seed, history_actions, planned_actions, goal_pos=None):
    """
    Ground-truth check: replay an episode's history in the real environment, then
    execute a planner's proposed action sequence, and check whether the given
    agent actually reaches the goal cell.

    Args:
        env: gym-interface MultiGrid env (already constructed, not yet reset for this run)
        agent_idx: index of the agent we're checking (int)
        seed: seed used to reset the env, so the replay matches the recorded episode
        history_actions: list of per-step action dicts (dict[agent_idx -> action]),
                         covering steps 0..t0 that actually happened before planning
        planned_actions: list of per-step action dicts covering the planner's
                         proposed future actions (this agent's plan + some choice
                         of partner actions -- see note below)
        goal_pos: optional (x, y) goal position; if None, read from env after reset

    Returns:
        dict with:
            reached: bool -- did the agent's position match goal_pos at any point
                     during planned_actions?
            reached_at_step: int or None -- offset within planned_actions where it
                     first happened (None if never)
            terminated_flag: bool -- did env's own termination fire for this agent
                     during planned_actions? (secondary signal, env-implementation
                     dependent)
    """
    obs, info = env.reset(seed=seed)
    if goal_pos is None:
        goal_pos = env.unwrapped.goal_pos

    # Replay the recorded history so the env state matches what the planner
    # actually conditioned on at t0.
    for step_actions in history_actions:
        obs, rewards, terminations, truncations, info = env.step(step_actions)

    reached = False
    reached_at_step = None
    terminated_flag = False

    for step_i, step_actions in enumerate(planned_actions):
        obs, rewards, terminations, truncations, info = env.step(step_actions)

        agent_pos = tuple(env.unwrapped.agents[agent_idx].pos)
        if agent_pos == tuple(goal_pos):
            reached = True
            reached_at_step = step_i
            terminated_flag = bool(terminations[agent_idx]) if hasattr(terminations, "__getitem__") else bool(terminations)
            break

        term_i = terminations[agent_idx] if hasattr(terminations, "__getitem__") else terminations
        if term_i:
            terminated_flag = True
            # Termination without matching goal_pos could mean a different
            # end condition (collision, timeout) -- don't assume success here.
            break

    return {
        "reached": reached,
        "reached_at_step": reached_at_step,
        "terminated_flag": terminated_flag,
    }

In [ ]:
#| export
def evaluate_planner_in_env(env, evaluator, episode, t0, seed, agent_order):
    """
    Full pipeline: get each agent's CEM plan at t0, then replay history + execute
    the joint plan in the real environment, checking goal-reach per agent.
    """
    plans, infos = get_all_agent_plans(evaluator, episode, t0)

    # Reconstruct joint action dicts for the HISTORY window (steps 0..t0), from
    # the recorded episode actions -- needed so the env state at t0 matches what
    # the planners actually conditioned on.
    history_actions = []
    H = evaluator.history_size
    for t in range(t0 - H + 1, t0 + 1):
        step_dict = {
            env_idx: int(episode[agent_key]["action"][0, t].item())
            for agent_key, env_idx in agent_order.items()
        }
        history_actions.append(step_dict)

    planned_actions = build_joint_step_actions(plans, list(agent_order.values()))

    results = {}
    for agent_key, env_idx in agent_order.items():
        result = check_agent_reached_goal(
            env, agent_idx=env_idx, seed=seed,
            history_actions=history_actions, planned_actions=planned_actions,
        )
        results[agent_key] = result

    return results

In [ ]:
#| export
def check_all_agents_reached_goal(env, agent_env_indices, seed, history_actions, planned_actions, goal_pos=None):
    obs, info = env.reset(seed=seed)
    if goal_pos is None:
        goal_pos = env.unwrapped.goal_pos

    for step_actions in history_actions:
        env.step(step_actions)

    reached = {idx: False for idx in agent_env_indices}
    reached_at_step = {idx: None for idx in agent_env_indices}

    for step_i, step_actions in enumerate(planned_actions):
        obs, rewards, terminations, truncations, info = env.step(step_actions)
        for idx in agent_env_indices:
            if reached[idx]:
                continue
            agent_pos = tuple(env.unwrapped.agents[idx].pos)
            if agent_pos == tuple(goal_pos):
                reached[idx] = True
                reached_at_step[idx] = step_i
        if all(reached.values()):
            break

    return {idx: {"reached": reached[idx], "reached_at_step": reached_at_step[idx]} for idx in agent_env_indices}

In [ ]:
#| hide
# @torch.no_grad()
# def _goal_error(self, episode, agent, t0, plan):
#     """Roll the *true* dynamics forward isn't available (no env here), so this
#     measures embedding-space distance between the model's own final predicted
#     state and the true goal embedding -- i.e. how well the model itself thinks
#     the plan reaches the goal. Swap for env-based execution if you have a sim.
#     """
#     H = self.history_size
#     pixels = episode[agent]["pixels"][t0 - H + 1 : t0 + 1].unsqueeze(0).to(self.device)
#     actions = episode[agent]["action"][t0 - H + 1 : t0 + 1].unsqueeze(0).to(self.device)
#     goal_pixels = episode[agent]["pixels"][t0 + self.goal_offset].unsqueeze(0).to(self.device)

#     info = {"pixels": pixels, "action": actions, "goal": goal_pixels}
#     action_seq = torch.cat([actions, plan.to(self.device)], dim=1).unsqueeze(1)  # add S=1 dim
#     info_for_cost = {k: (v.unsqueeze(1) if torch.is_tensor(v) else v) for k, v in info.items()}
#     cost = self.model.get_cost(info_for_cost, action_seq)  # (1, 1)
#     return cost.item()


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()